In [ ]:
# Remove coment if the script is run from the jupyter notebook instead of the terminal
# !pip install -q papermill

In [ ]:
import os
import time
import papermill as pm

# Directory to save the executed notebooks
notebook_output = "notebook_run_output"
os.makedirs(notebook_output, exist_ok=True)

# Experiment variants (will be pass as parameter)
variants = [
    {"code": 1, "name": "baseline"},
    {"code": 2, "name": "without_id"},
    {"code": 3, "name": "without_month_snapshot"},
    {"code": 4, "name": "without_both"}
]

# All seeds (will be pass as parameter as well)
seeds = [
    100019, 100043, 100049, 100057, 100069,
    323579, 523597, 723587, 823591, 923579
]

notebook_template = "worker.ipynb" # Name of the notebook to execute
total_runs = len(variants) * len(seeds)
run_num = 1

t_start_total = time.time()

for var in variants:
    for idx, sem in enumerate(seeds, start=1):
        exp_id = f"E16v2{var['code']}{idx:02d}"
        output_path = f"{notebook_output}/run_{exp_id}.ipynb"
        exp_directory = f"/content/buckets/b1/exp/WFE16v2{var['code']}{idx:02d}"
        
        # Omit run if the PARAM.yml file is already there
        if os.path.exists(f"{exp_directory}/PARAM.yml"):
            print(f"[{run_num}/{total_runs}] Omitting {exp_id} ({var['name']} - Seed {sem}) already ran.")
            run_num += 1
            continue
            
        print(f"\n========================================================")
        print(f"[{run_num}/{total_runs}] Initializing: {exp_id} | Variant: {var['name']} | Seed: {sem}")
        print(f"========================================================")
        
        t0 = time.time()
        try:
            pm.execute_notebook(
                input_path=notebook_template,
                output_path=output_path,
                parameters={
                    "k_variant_code": var["code"],
                    "k_seed_val": sem,
                    "k_seed_idx": idx
                },
                progress_bar=False,
                kernel_name="ir"  # Kernel de R
            )
            print(f"✓ Success {exp_id} in {(time.time() - t0)/60:.2f} minutes.")
        except Exception as e:
            print(f"✗ ERROR on run {exp_id}: {e}")
            
        run_num += 1

print(f"\nProcess ended. Total accumulated time: {(time.time() - t_start_total)/3600:.2f} hours.")